In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

In [0]:
import yaml
CONFIG_PATH = "/Workspace/Users/avranilset@gmail.com/capstone-project/config/config.yaml"

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

# Databricks config
CATALOG = config["databricks"]["catalog"]
BRONZE_SCHEMA = config["databricks"]["schemas"]["bronze"]
RAW_DATA_PATH = config["databricks"]["raw_data_path"]

# Dataset config
datasets = config["datasets"]

print("Config loaded successfully")

In [0]:
from pyspark.sql.functions import current_timestamp, lit

def ingest_csv_to_bronze(
    dataset_key,
    schema,
    datasets_config,
    raw_path,
    catalog,
    bronze_schema,
    counts_dict
):
    print(f"\nIngesting {dataset_key}.csv")

    table_name = datasets_config[dataset_key]["bronze_table"]
    file_name = datasets_config[dataset_key]["filename"]

    df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .schema(schema)
        .load(f"{raw_path}{file_name}")
    )

    df = (
        df.withColumn("ingestion_timestamp", current_timestamp())
          .withColumn("source_file_name", lit(file_name))
    )

    count_val = df.count()

    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{catalog}.{bronze_schema}.{table_name}")

    print(f" {catalog}.{bronze_schema}.{table_name}: {count_val:,} rows")

    counts_dict[table_name] = count_val

In [0]:
customers_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("country", StringType(), True),
])

products_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
])

orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("status", StringType(), True),
])

order_items_schema = StructType([
    StructField("order_item_id", IntegerType(), True),
    StructField("order_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
])


print("BRONZE INGESTION — START")


counts = {}

In [0]:
ingest_csv_to_bronze(
    "customers",
    customers_schema,
    datasets,
    RAW_DATA_PATH,
    CATALOG,
    BRONZE_SCHEMA,
    counts
)

ingest_csv_to_bronze(
    "products",
    products_schema,
    datasets,
    RAW_DATA_PATH,
    CATALOG,
    BRONZE_SCHEMA,
    counts
)

ingest_csv_to_bronze(
    "orders",
    orders_schema,
    datasets,
    RAW_DATA_PATH,
    CATALOG,
    BRONZE_SCHEMA,
    counts
)

ingest_csv_to_bronze(
    "order_items",
    order_items_schema,
    datasets,
    RAW_DATA_PATH,
    CATALOG,
    BRONZE_SCHEMA,
    counts
)

In [0]:

print(f"BRONZE INGESTION — COMPLETE | {counts}")
